# Janus Valuation PE/PB｜獨立補收研究 Notebook v1

**僅研究資料工程**。預設 `DRY_RUN=True`、`RUN_FETCH=False`；沒有全市場補收、訓練、發布、部署或 canonical 指標修改。必要時自行**明確**開啟有界來源 smoke。

權威程式版本：`research/colab/valuation_backfill.py`（下方 SHA256 綁定）；Drive 僅執行 GitHub 對應已同步檔，不內嵌第二套模組。發生 hash 不一致必須停止。

## 1. 固定版本、用途與限制

In [ ]:
MODULE_SHA256='c7d05cda416dc64505374f317f7970c31a3efbc509bfe3259025a55d5b26e1dd'
PUBLICATION_ALLOWED=False
print('Valuation backfill module sha256:',MODULE_SHA256)

## 2. 安裝固定版本／相容依賴

In [ ]:
import sys,subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','pandas==2.2.3','numpy==2.2.6','pyarrow==20.0.0','requests==2.32.5'])

## 3. 掛載 Drive、讀 secrets、確認唯一 ROOT

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import os,sys,hashlib
ROOT=Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
if not ROOT.is_dir():raise FileNotFoundError(f'白名單 ROOT 不存在：{ROOT}')
MODULE=ROOT/'CoLab程式'/'valuation_backfill.py'
if not MODULE.is_file():raise FileNotFoundError(f'先將 GitHub 相同模組同步至既有 CoLab程式：{MODULE}')
actual=hashlib.sha256(MODULE.read_bytes()).hexdigest()
if actual!=MODULE_SHA256:raise RuntimeError('Drive 模組 SHA256 不符；不得執行漂移版本')
if str(MODULE.parent) not in sys.path:sys.path.insert(0,str(MODULE.parent))
import valuation_backfill as v
v.checked_root(ROOT)
try: token=userdata.get('FINMIND_TOKEN')
except Exception:token=None
token=token or os.environ.get('FINMIND_TOKEN')
print('ROOT validated; secret configured:',bool(token))

## 4. 保守範圍與有界請求預算

In [ ]:
RUN_ID='valuation-backfill-research-v1'
SYMBOLS=['2330','2882','6488']  # 上市非金融、金融、上櫃非金融；自行核對最新普通股母體
MARKETS={'2330':'twse','2882':'twse','6488':'tpex'}
START='2025-01-01';END='2026-09-30'
DRY_RUN=True;RUN_FETCH=False
MAX_REQUESTS=3;MAX_SYMBOLS=3;MAX_RUNTIME_MINUTES=5;MAX_RETRIES=2;REQUEST_INTERVAL=1.0
run=ROOT/'模型研究'/'valuation'/'backfill_candidates'/RUN_ID
# 只在已確認白名單 ROOT 內建立 candidate run；不建立第二套資料根。
run.mkdir(parents=True,exist_ok=True)
print('run:',run)

## 5. Inventory：掃描 CURRENT、manifest 與所有 referenced Parquet，驗 SHA

In [ ]:
rec=v.inventory(ROOT,run)
print('observations:',len(rec['inventory']))

## 6. Plan：依資料實際可用性分類與估計 HTTP 請求

In [ ]:
tasks=v.plan(ROOT,run,rec,symbols=SYMBOLS,start=START,end=END,markets=MARKETS)
print('planned HTTP requests:',len(tasks))
import pandas as pd
display(pd.read_csv(run/'gap_plan.csv').head(15))

## 7. Fetch：預設關閉；僅人工修改 RUN_FETCH 與 DRY_RUN 後有界呼叫

In [ ]:
receipt=v.fetch(ROOT,run,run_fetch=RUN_FETCH,dry_run=DRY_RUN,max_requests=MAX_REQUESTS,max_symbols=MAX_SYMBOLS,max_runtime_minutes=MAX_RUNTIME_MINUTES,max_retries=MAX_RETRIES,request_interval=REQUEST_INTERVAL,token=token)
print(receipt)

## 8. Normalize：保留 raw，未知口徑不能自動合格

In [ ]:
observations=v.normalize(run)
print('normalized source rows:',len(observations))

## 9. Derive：僅讀研究候選的已驗證觀察，不對網路請求

In [ ]:
factors=v.derive(run)
print('safe derived factors:',len(factors))

## 10. Validate：檢查 CURRENT 未漂移、完整 marker／hash；產生 candidate manifest

In [ ]:
manifest=v.validate(ROOT,run)
print(manifest['state'],'issues:',manifest['quality_issues'])

## 11. Report：輸出缺口、進度、來源及 PIT 說明

In [ ]:
print(v.report(run))
print('reports:',sorted(x.name for x in run.iterdir() if x.is_file()))

## 12. Resume：中斷後重新從第 3 格開始

使用原 `RUN_ID` 與完全相同的選股和期間，依序執行第 3～11 格；`SUCCESS` 必須通過完整 marker、檔案 hash、schema／symbol 與 row count 讀回才略過。若遇到 `writer.lock`，先確認舊 Colab 已停止，人工審查並處理 stale lock，不自動接管。不同 `ROOT` 或 CURRENT SHA 應建立新的 RUN_ID。所有來源與 PIT 未完成項均保留 `NOT_VERIFIED`，不得當成模型可用證據。